### Notebook 07 — Walk-Forward Validation

**Goal:** Implement rigorous Walk-Forward Validation (Time-Series Cross Validation) to evaluate our 2-lag and 12-lag models realistically.


### Imports


In [1]:
from __future__ import annotations

from pathlib import Path
import os
import warnings
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import ElasticNet

import mlflow
import mlflow.sklearn

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
warnings.filterwarnings("ignore")


/media/breezy/NewVolume/projects_int/dengue/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Configuration


In [2]:
DATA_DIR = Path("../data/processed")

FEATURE_PATH_1LAGS = DATA_DIR / "dengue_features_1lags.parquet"
FEATURE_PATH_2LAGS = DATA_DIR / "dengue_features_2lags.parquet"
FEATURE_PATH_4LAGS = DATA_DIR / "dengue_features_4lags.parquet"
FEATURE_PATH_12LAGS = DATA_DIR / "dengue_features_12lags.parquet"

MLFLOW_DIR = Path("../mlruns")

RESULTS_DIR = Path("../data/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = RESULTS_DIR / "notebook07_walk_forward_results.parquet"

EXPERIMENT_NAME = "dengue_walk_forward_validation"


### Configure MLflow


In [3]:
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(MLFLOW_DIR.resolve().as_uri())
mlflow.set_experiment(EXPERIMENT_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT_NAME)


MLflow tracking URI: file:///media/breezy/NewVolume/projects_int/dengue/mlruns
Experiment: dengue_walk_forward_validation


### Load Datasets


In [4]:
df_1 = pd.read_parquet(FEATURE_PATH_1LAGS).sort_values(["week_start", "district"]).reset_index(drop=True)
df_2 = pd.read_parquet(FEATURE_PATH_2LAGS).sort_values(["week_start", "district"]).reset_index(drop=True)
df_4 = pd.read_parquet(FEATURE_PATH_4LAGS).sort_values(["week_start", "district"]).reset_index(drop=True)
df_12 = pd.read_parquet(FEATURE_PATH_12LAGS).sort_values(["week_start", "district"]).reset_index(drop=True)

print("1-lag shape:", df_1.shape)
print("2-lag shape:", df_2.shape)
print("4-lag shape:", df_4.shape)
print("12-lag shape:", df_12.shape)


1-lag shape: (26338, 20)
2-lag shape: (26286, 25)
4-lag shape: (26182, 30)
12-lag shape: (25766, 34)


### Define features


In [5]:
feature_columns_1 = [c for c in df_1.columns if c not in ["district", "week_start", "cases"]]
feature_columns_2 = [c for c in df_2.columns if c not in ["district", "week_start", "cases"]]
feature_columns_4 = [c for c in df_4.columns if c not in ["district", "week_start", "cases"]]
feature_columns_12 = [c for c in df_12.columns if c not in ["district", "week_start", "cases"]]

print(f"1-lag feature count: {len(feature_columns_1)}")
print(f"2-lag feature count: {len(feature_columns_2)}")
print(f"4-lag feature count: {len(feature_columns_4)}")
print(f"12-lag feature count: {len(feature_columns_12)}")


1-lag feature count: 17
2-lag feature count: 22
4-lag feature count: 27
12-lag feature count: 31


### Time-Series Split Generator

Because we have panel data (multiple districts for each date), we split on the **unique dates**, not the individual rows.


In [6]:
def generate_walk_forward_splits(df: pd.DataFrame, n_splits=5):
    # Extract unique sorted dates
    unique_dates = np.sort(df["week_start"].dropna().unique())
    
    tscv = TimeSeriesSplit(n_splits=n_splits)
    
    splits = []
    for fold_idx, (train_date_idx, test_date_idx) in enumerate(tscv.split(unique_dates)):
        train_dates = unique_dates[train_date_idx]
        test_dates = unique_dates[test_date_idx]
        
        train_df = df[df["week_start"].isin(train_dates)].copy()
        test_df = df[df["week_start"].isin(test_dates)].copy()
        
        # Security check to prevent leakage
        assert train_df["week_start"].max() < test_df["week_start"].min()
        
        splits.append({
            "fold": fold_idx + 1,
            "train_df": train_df,
            "test_df": test_df,
            "train_dates": (train_dates.min(), train_dates.max()),
            "test_dates": (test_dates.min(), test_dates.max())
        })
        
    return splits

splits_2 = generate_walk_forward_splits(df_2, n_splits=5)
splits_12 = generate_walk_forward_splits(df_12, n_splits=5)
splits_1 = generate_walk_forward_splits(df_1, n_splits=5)


### Review Fold Boundaries


In [7]:
for split in splits_2:
    print(f"Fold {split['fold']}:")
    print(f"  Train: {str(split['train_dates'][0])[:10]} to {str(split['train_dates'][1])[:10]} (Rows: {len(split['train_df'])})")
    print(f"  Test:  {str(split['test_dates'][0])[:10]} to {str(split['test_dates'][1])[:10]} (Rows: {len(split['test_df'])})")
    print()


Fold 1:
  Train: 2007-01-15 to 2010-04-19 (Rows: 4446)
  Test:  2010-04-26 to 2013-07-08 (Rows: 4368)

Fold 2:
  Train: 2007-01-15 to 2013-07-08 (Rows: 8814)
  Test:  2013-07-15 to 2016-10-03 (Rows: 4368)

Fold 3:
  Train: 2007-01-15 to 2016-10-03 (Rows: 13182)
  Test:  2016-10-10 to 2019-12-23 (Rows: 4368)

Fold 4:
  Train: 2007-01-15 to 2019-12-23 (Rows: 17550)
  Test:  2019-12-30 to 2023-04-10 (Rows: 4368)

Fold 5:
  Train: 2007-01-15 to 2023-04-10 (Rows: 21918)
  Test:  2023-04-17 to 2026-06-29 (Rows: 4368)



### Cross-Validation Evaluation Loop


In [8]:
def evaluate_model_across_folds(splits, feature_columns, model, run_name_prefix):
    metrics_list = []
    for split in splits:
        fold = split["fold"]
        X_train = split["train_df"][feature_columns]
        y_train = split["train_df"]["cases"]
        X_test = split["test_df"][feature_columns]
        y_test = split["test_df"]["cases"]
        
        with mlflow.start_run(run_name=f"{run_name_prefix}_Fold_{fold}"):
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            y_pred = np.clip(y_pred, 0, None)
            
            mae = mean_absolute_error(y_test, y_pred)
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            r2 = r2_score(y_test, y_pred)
            
            mlflow.log_param("fold", fold)
            mlflow.log_param("n_features", len(feature_columns))
            mlflow.log_metric("MAE", mae)
            mlflow.log_metric("RMSE", rmse)
            mlflow.log_metric("R2", r2)
            
            metrics_list.append({
                "model": run_name_prefix,
                "fold": fold,
                "MAE": mae,
                "RMSE": rmse,
                "R2": r2
            })
    return pd.DataFrame(metrics_list)


### Define Models


In [9]:
dummy_model = DummyRegressor(strategy="mean")

elastic_model = ElasticNet(random_state=42)

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1
)

lgb_model = LGBMRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)


# Optuna Tuned LightGBM
lgb_tuned_model = LGBMRegressor(
    n_estimators=350,
    max_depth=5,
    num_leaves=115,
    learning_rate=0.04,
    min_child_samples=14,
    subsample=0.91,
    colsample_bytree=0.87,
    reg_alpha=0.10,
    reg_lambda=3.95,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)


### Dummy on 1Lag


In [10]:
res_dummy_1lag = evaluate_model_across_folds(
    splits=splits_2, 
    feature_columns=feature_columns_1, 
    model=dummy_model, 
    run_name_prefix="1Lag_Dummy"
)
display(res_dummy_1lag)


,model,fold,MAE,RMSE,R2
0,1Lag_Dummy,1,15.621151,32.863088,-0.035585
1,1Lag_Dummy,2,20.252769,47.477424,-0.030487
2,1Lag_Dummy,3,65.396965,166.500534,-0.138156
3,1Lag_Dummy,4,29.968893,53.292424,-0.011061
4,1Lag_Dummy,5,41.156705,87.518220,-0.046772


### Dummy on 2Lag


In [11]:
res_dummy_2lag = evaluate_model_across_folds(
    splits=splits_2, 
    feature_columns=feature_columns_2, 
    model=dummy_model, 
    run_name_prefix="2Lag_Dummy"
)
display(res_dummy_2lag)


,model,fold,MAE,RMSE,R2
0,2Lag_Dummy,1,15.621151,32.863088,-0.035585
1,2Lag_Dummy,2,20.252769,47.477424,-0.030487
2,2Lag_Dummy,3,65.396965,166.500534,-0.138156
3,2Lag_Dummy,4,29.968893,53.292424,-0.011061
4,2Lag_Dummy,5,41.156705,87.518220,-0.046772


### Dummy on 12Lag


In [12]:
res_dummy_12lag = evaluate_model_across_folds(
    splits=splits_12, 
    feature_columns=feature_columns_12, 
    model=dummy_model, 
    run_name_prefix="12Lag_Dummy"
)
display(res_dummy_12lag)


,model,fold,MAE,RMSE,R2
0,12Lag_Dummy,1,15.969018,33.288852,-0.031075
1,12Lag_Dummy,2,20.391460,47.588737,-0.028004
2,12Lag_Dummy,3,63.015881,163.420146,-0.129160
3,12Lag_Dummy,4,32.751750,65.141340,-0.000837
4,12Lag_Dummy,5,41.236606,87.817951,-0.044624


### ElasticNet on 1Lag


In [13]:
res_elasticnet_1lag = evaluate_model_across_folds(
    splits=splits_2, 
    feature_columns=feature_columns_1, 
    model=elastic_model, 
    run_name_prefix="1Lag_ElasticNet"
)
display(res_elasticnet_1lag)


,model,fold,MAE,RMSE,R2
0,1Lag_ElasticNet,1,7.906116,17.816693,0.695615
1,1Lag_ElasticNet,2,8.466289,18.287754,0.847107
2,1Lag_ElasticNet,3,19.887195,50.553148,0.895078
3,1Lag_ElasticNet,4,12.745901,46.726735,0.222721
4,1Lag_ElasticNet,5,15.476796,48.840735,0.673999


### ElasticNet on 2Lag


In [14]:
res_elasticnet_2lag = evaluate_model_across_folds(
    splits=splits_2, 
    feature_columns=feature_columns_2, 
    model=elastic_model, 
    run_name_prefix="2Lag_ElasticNet"
)
display(res_elasticnet_2lag)


,model,fold,MAE,RMSE,R2
0,2Lag_ElasticNet,1,7.534565,16.820680,0.728696
1,2Lag_ElasticNet,2,7.927794,17.531314,0.859493
2,2Lag_ElasticNet,3,19.682887,51.774728,0.889946
3,2Lag_ElasticNet,4,12.583530,45.025407,0.278292
4,2Lag_ElasticNet,5,15.012356,46.164183,0.708750


### ElasticNet on 12Lag


In [15]:
res_elasticnet_12lag = evaluate_model_across_folds(
    splits=splits_12, 
    feature_columns=feature_columns_12, 
    model=elastic_model, 
    run_name_prefix="12Lag_ElasticNet"
)
display(res_elasticnet_12lag)


,model,fold,MAE,RMSE,R2
0,12Lag_ElasticNet,1,7.488778,16.583982,0.744100
1,12Lag_ElasticNet,2,7.945026,17.692171,0.857915
2,12Lag_ElasticNet,3,19.748174,53.825257,0.877506
3,12Lag_ElasticNet,4,13.621743,46.669838,0.486285
4,12Lag_ElasticNet,5,15.171738,46.676853,0.704881


### RF on 1Lag


In [16]:
res_rf_1lag = evaluate_model_across_folds(
    splits=splits_2, 
    feature_columns=feature_columns_1, 
    model=rf_model, 
    run_name_prefix="1Lag_RF"
)
display(res_rf_1lag)


,model,fold,MAE,RMSE,R2
0,1Lag_RF,1,8.483072,19.346628,0.641095
1,1Lag_RF,2,9.246557,21.242367,0.793712
2,1Lag_RF,3,27.057062,100.147782,0.588232
3,1Lag_RF,4,12.734826,45.880648,0.250614
4,1Lag_RF,5,16.708161,53.590719,0.607505


### RF on 2Lag


In [17]:
res_rf_2lag = evaluate_model_across_folds(
    splits=splits_2, 
    feature_columns=feature_columns_2, 
    model=rf_model, 
    run_name_prefix="2Lag_RF"
)
display(res_rf_2lag)


,model,fold,MAE,RMSE,R2
0,2Lag_RF,1,8.068491,18.318253,0.678236
1,2Lag_RF,2,8.516462,20.363163,0.810435
2,2Lag_RF,3,26.283938,100.419624,0.585993
3,2Lag_RF,4,12.909187,46.089286,0.243783
4,2Lag_RF,5,15.105460,45.124768,0.721718


### RF on 12Lag


In [18]:
res_rf_12lag = evaluate_model_across_folds(
    splits=splits_12, 
    feature_columns=feature_columns_12, 
    model=rf_model, 
    run_name_prefix="12Lag_RF"
)
display(res_rf_12lag)


,model,fold,MAE,RMSE,R2
0,12Lag_RF,1,7.848638,17.943742,0.700416
1,12Lag_RF,2,8.427844,20.004067,0.818355
2,12Lag_RF,3,26.008292,101.878356,0.561158
3,12Lag_RF,4,14.251555,47.814355,0.460780
4,12Lag_RF,5,14.815301,45.437215,0.720349


### XGB on 1Lag


In [19]:
res_xgb_1lag = evaluate_model_across_folds(
    splits=splits_2, 
    feature_columns=feature_columns_1, 
    model=xgb_model, 
    run_name_prefix="1Lag_XGB"
)
display(res_xgb_1lag)


,model,fold,MAE,RMSE,R2
0,1Lag_XGB,1,8.527288,19.474029,0.636353
1,1Lag_XGB,2,9.585520,23.007071,0.758014
2,1Lag_XGB,3,28.634117,102.222562,0.570993
3,1Lag_XGB,4,12.898707,45.129040,0.274966
4,1Lag_XGB,5,17.199354,55.940583,0.572330


### XGB on 2Lag


In [20]:
res_xgb_2lag = evaluate_model_across_folds(
    splits=splits_2, 
    feature_columns=feature_columns_2, 
    model=xgb_model, 
    run_name_prefix="2Lag_XGB"
)
display(res_xgb_2lag)


,model,fold,MAE,RMSE,R2
0,2Lag_XGB,1,8.088881,18.349179,0.677149
1,2Lag_XGB,2,8.846739,21.067965,0.797085
2,2Lag_XGB,3,27.243589,100.414634,0.586034
3,2Lag_XGB,4,12.796313,45.072359,0.276786
4,2Lag_XGB,5,15.813287,48.166703,0.682935


### XGB on 12Lag


In [21]:
res_xgb_12lag = evaluate_model_across_folds(
    splits=splits_12, 
    feature_columns=feature_columns_12, 
    model=xgb_model, 
    run_name_prefix="12Lag_XGB"
)
display(res_xgb_12lag)


,model,fold,MAE,RMSE,R2
0,12Lag_XGB,1,8.114669,18.708334,0.674341
1,12Lag_XGB,2,8.507578,20.189701,0.814968
2,12Lag_XGB,3,27.365341,107.685197,0.509706
3,12Lag_XGB,4,14.555264,46.637029,0.487007
4,12Lag_XGB,5,15.868713,51.100559,0.646292


### LGB on 1Lag


In [22]:
res_lgb_1lag = evaluate_model_across_folds(
    splits=splits_2, 
    feature_columns=feature_columns_1, 
    model=lgb_model, 
    run_name_prefix="1Lag_LGB"
)
display(res_lgb_1lag)


,model,fold,MAE,RMSE,R2
0,1Lag_LGB,1,8.518094,19.520674,0.634608
1,1Lag_LGB,2,9.683872,23.923734,0.738347
2,1Lag_LGB,3,28.194664,102.154748,0.571562
3,1Lag_LGB,4,12.723841,46.248835,0.238539
4,1Lag_LGB,5,16.911974,52.279703,0.626474


### LGB on 2Lag


In [23]:
res_lgb_2lag = evaluate_model_across_folds(
    splits=splits_2, 
    feature_columns=feature_columns_2, 
    model=lgb_model, 
    run_name_prefix="2Lag_LGB"
)
display(res_lgb_2lag)


,model,fold,MAE,RMSE,R2
0,2Lag_LGB,1,8.117198,18.621206,0.667505
1,2Lag_LGB,2,8.937366,22.090213,0.776916
2,2Lag_LGB,3,27.118550,101.348617,0.578298
3,2Lag_LGB,4,12.691186,44.845799,0.284038
4,2Lag_LGB,5,15.260906,46.545054,0.703925


### LGB on 12Lag


In [24]:
res_lgb_12lag = evaluate_model_across_folds(
    splits=splits_12, 
    feature_columns=feature_columns_12, 
    model=lgb_model, 
    run_name_prefix="12Lag_LGB"
)
display(res_lgb_12lag)


,model,fold,MAE,RMSE,R2
0,12Lag_LGB,1,8.024164,18.567724,0.679218
1,12Lag_LGB,2,8.543635,20.633297,0.806748
2,12Lag_LGB,3,26.720142,101.662622,0.563015
3,12Lag_LGB,4,13.839479,46.399416,0.492221
4,12Lag_LGB,5,15.388787,48.407998,0.682585


### Summary & Conclusion


### Tuned LightGBM on 2Lag


In [25]:
res_lgb_tuned_2lag = evaluate_model_across_folds(
    splits=splits_2,
    feature_columns=feature_columns_2,
    model=lgb_tuned_model,
    run_name_prefix="2Lag_LGB_Tuned"
)
display(res_lgb_tuned_2lag)


,model,fold,MAE,RMSE,R2
0,2Lag_LGB_Tuned,1,8.066040,18.526328,0.670885
1,2Lag_LGB_Tuned,2,8.901414,22.378165,0.771062
2,2Lag_LGB_Tuned,3,26.449556,100.187532,0.587905
3,2Lag_LGB_Tuned,4,12.636873,43.923045,0.313199
4,2Lag_LGB_Tuned,5,15.260312,45.725199,0.714263


### Tuned LightGBM on 12Lag


In [26]:
res_lgb_tuned_12lag = evaluate_model_across_folds(
    splits=splits_12,
    feature_columns=feature_columns_12,
    model=lgb_tuned_model,
    run_name_prefix="12Lag_LGB_Tuned"
)
display(res_lgb_tuned_12lag)


,model,fold,MAE,RMSE,R2
0,12Lag_LGB_Tuned,1,7.946254,18.485592,0.682050
1,12Lag_LGB_Tuned,2,8.545840,20.796412,0.803681
2,12Lag_LGB_Tuned,3,26.030544,99.973712,0.577413
3,12Lag_LGB_Tuned,4,14.141894,46.142872,0.497820
4,12Lag_LGB_Tuned,5,15.065502,45.236126,0.722818


In [27]:
all_results = pd.concat([
    res_dummy_1lag, res_dummy_2lag, res_dummy_12lag,
    res_elasticnet_1lag, res_elasticnet_2lag, res_elasticnet_12lag,
    res_rf_1lag, res_rf_2lag, res_rf_12lag,
    res_xgb_1lag, res_xgb_2lag, res_xgb_12lag,
    res_lgb_1lag, res_lgb_2lag, res_lgb_12lag,
    res_lgb_tuned_2lag, res_lgb_tuned_12lag,
])

summary = all_results.groupby("model").agg(
    MAE_mean=("MAE", "mean"),
    MAE_std=("MAE", "std"),
    RMSE_mean=("RMSE", "mean"),
    R2_mean=("R2", "mean"),
).sort_values("MAE_mean")

display(summary)

# Save results
all_results.to_parquet(RESULTS_PATH)
print("Saved walk-forward results to:", RESULTS_PATH)


,MAE_mean,MAE_std,RMSE_mean,R2_mean
model,,,,
2Lag_ElasticNet,12.548227,5.085808,35.463262,0.693036
12Lag_ElasticNet,12.795092,5.156483,36.289620,0.734137
1Lag_ElasticNet,12.896459,5.002028,36.445013,0.666904
2Lag_RF,14.176708,7.389231,46.063019,0.608033
2Lag_LGB_Tuned,14.262839,7.404390,46.148054,0.611463
12Lag_RF,14.270326,7.304962,46.615547,0.652211
12Lag_LGB_Tuned,14.346007,7.274725,46.126943,0.656756
2Lag_LGB,14.425041,7.660508,46.690178,0.602136
12Lag_LGB,14.503241,7.549384,47.134212,0.644757


Saved walk-forward results to: ../data/results/notebook07_walk_forward_results.parquet


In [28]:
print(summary)

                   MAE_mean    MAE_std  RMSE_mean   R2_mean
model                                                      
2Lag_ElasticNet   12.548227   5.085808  35.463262  0.693036
12Lag_ElasticNet  12.795092   5.156483  36.289620  0.734137
1Lag_ElasticNet   12.896459   5.002028  36.445013  0.666904
2Lag_RF           14.176708   7.389231  46.063019  0.608033
2Lag_LGB_Tuned    14.262839   7.404390  46.148054  0.611463
12Lag_RF          14.270326   7.304962  46.615547  0.652211
12Lag_LGB_Tuned   14.346007   7.274725  46.126943  0.656756
2Lag_LGB          14.425041   7.660508  46.690178  0.602136
12Lag_LGB         14.503241   7.549384  47.134212  0.644757
2Lag_XGB          14.557762   7.747120  46.614168  0.603998
1Lag_RF           14.845935   7.564314  48.041629  0.576232
12Lag_XGB         14.882313   7.799747  48.864164  0.626463
1Lag_LGB          15.206489   7.952969  48.825539  0.561906
1Lag_XGB          15.368997   8.149747  49.154657  0.562531
2Lag_Dummy        34.479297  19.867420  